In [2]:
from transformers import MarianMTModel, MarianTokenizer
import pandas as pd
from tqdm import tqdm
import torch

# 加载预训练翻译模型和tokenizer
model_name = "/home/wangshuo/resource/AIModels/NLP/translate/opus-mt-en-zh"  # 你本地的路径
model = MarianMTModel.from_pretrained(model_name)
tokenizer = MarianTokenizer.from_pretrained(model_name)

# 检查是否有可用的GPU，并将模型转移到GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 批量翻译函数
def batch_translate(texts, batch_size=32):
    # 将文本分为多个批次
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="翻译进度", unit="批次"):
        batch = texts[i:i+batch_size]
        
        # 确保 batch 是一个字符串的列表
        batch = [str(text) for text in batch]  # 将每个元素确保转换为字符串
        
        # 使用tokenizer进行批量编码
        translated = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        
        # 将数据移动到GPU（如果有的话）
        translated = {key: value.to(device) for key, value in translated.items()}
        
        # 使用模型进行批量翻译
        translated_ids = model.generate(**translated)
        
        # 解码翻译结果
        translated_texts = [tokenizer.decode(t, skip_special_tokens=True) for t in translated_ids]
        translations.extend(translated_texts)
    
    return translations

# 读取posts.csv文件
dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_2/'
df = pd.read_csv(dir + 'posts_with_nli_results_1.csv')

# 批量翻译并添加到新的列 body_translate
texts = df['body'].tolist()  # 将所有body字段转为列表
translated_texts = batch_translate(texts)  # 批量翻译

# 将翻译结果添加到DataFrame的新列
df['bio'] = translated_texts

# 保存新的csv文件
df.to_csv(dir + 'posts_with_nli_results_1_translated.csv', index=False)

print("翻译完成，结果已保存到 'posts_with_nli_results_1_translated.csv' 文件中")


翻译进度: 100%|██████████| 27/27 [04:45<00:00, 10.56s/批次]

翻译完成，结果已保存到 'posts_with_nli_results_1_translated.csv' 文件中


posts.csv内容提取：id:ID,nli_label，body_translate，pcNum这四个字段按顺序写入新文件

In [6]:
import csv

# 打开原始文件和目标文件
with open(dir + 'posts_with_nli_results_1_translated.csv', 'r', encoding='utf-8') as infile, open(dir + 'posts_with_nli_results_1_translated_test.csv', 'w', newline='', encoding='utf-8') as outfile:
    reader = csv.DictReader(infile)  # 使用字典读取方式
    fieldnames = ['id', 'nli_label', 'bio','body']
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    
    writer.writeheader()  # 写入表头

    # 提取每一行的相关字段并写入新文件
    for row in reader:
        writer.writerow({
            'id': row['id'],
            'nli_label': row['nli_label'],
            'bio': row['bio'],
            'body': row['body']
        })

print("数据已成功提取并写入 posts_with_nli_results_1_translated_test.csv")


数据已成功提取并写入 posts_with_nli_results_1_translated_test.csv


In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import pandas as pd
from tqdm import tqdm

# 加载预训练翻译模型和tokenizer
model_name = "/home/wangshuo/resource/AIModels/NLP/translate/opus-mt-en-zh"  # 你本地的路径
model = MarianMTModel.from_pretrained(model_name)
tokenizer = MarianTokenizer.from_pretrained(model_name)

# 检查是否有可用的GPU，并将模型转移到GPU
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 批量翻译函数
def batch_translate(texts, batch_size=32):
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="翻译进度", unit="批次"):
        batch = texts[i:i+batch_size]
        
        # 确保 batch 是一个字符串的列表
        batch = [str(text) for text in batch]
        
        # 使用tokenizer进行批量编码
        translated = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        
        # 将数据移动到GPU（如果有的话）
        translated = {key: value.to(device) for key, value in translated.items()}
        
        # 使用模型进行批量翻译
        translated_ids = model.generate(**translated)
        
        # 解码翻译结果
        translated_texts = [tokenizer.decode(t, skip_special_tokens=True) for t in translated_ids]
        translations.extend(translated_texts)
    
    return translations

# 读取comments.csv文件
dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_1/'
df = pd.read_csv(dir + 'comments.csv')

# 筛选 nli_label 为 'bart_entailment' 的行
filtered_df = df[df['nli_label'] == 'bart_entailment']

# 批量翻译并添加到新的列 body_trans
texts = filtered_df['body'].tolist()  # 将所有body字段转为列表
translated_texts = batch_translate(texts)  # 批量翻译

# 将翻译结果添加到DataFrame的新列
filtered_df['body_trans'] = translated_texts

# 只保留 'id', 'comments', 'body_trans' 列
result_df = filtered_df[['id:ID', 'comments', 'body_trans' ,'bart_entailment_probability']]

# 保存新的csv文件
result_df.to_csv(dir + 'comments_translated.csv', index=False)

print("翻译完成，结果已保存到 'comments_translated.csv' 文件中")